In [1]:
!pip install transformers torch datasets rouge-score bert-score -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 887.8 kB/s eta 0:00:00


In [2]:
from transformers import BartForConditionalGeneration, BartTokenizer
from transformers import T5ForConditionalGeneration, T5Tokenizer

print("Loading BART...")
bart_tokenizer = BartTokenizer.from_pretrained("facebook/bart-large-cnn")
bart_model = BartForConditionalGeneration.from_pretrained("facebook/bart-large-cnn")
print("BART loaded!")

print("Loading T5...")
t5_tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base")
t5_model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base")
print("T5 loaded!")

Loading BART...


vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

T5 loaded!


In [3]:
import re

def clean_text(text):
    text = re.sub(r'\[\d+\]', '', text)
    text = re.sub(r'\(\w+ et al\.,?\s*\d{4}\)', '', text)
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    return text

def truncate_text(text, max_words=800):
    words = text.split()
    if len(words) > max_words:
        words = words[:max_words]
    return ' '.join(words)

papers = [
    {"id": "paper_01", "field": "medicine", "title": "Development of mRNA Vaccines and Their Delivery System", "url": "https://pubmed.ncbi.nlm.nih.gov/36697236/", "abstract": "The rapid development of mRNA vaccines contributed to managing the COVID-19 pandemic.", "text": "The rapid development of mRNA vaccines contributed to managing the COVID-19 pandemic and may help manage future outbreaks of infectious diseases. Because the antigens targeted by mRNA vaccines can be easily altered by simply changing the sequence present in the coding region of mRNA structures it is more appropriate to develop vaccines especially during rapidly developing outbreaks. mRNA vaccines have great potential in inducing successful antigen-specific immunity by expressing target antigens in cells and simultaneously triggering immune responses."},
    {"id": "paper_02", "field": "medicine", "title": "SARS-CoV-2 mRNA Vaccines Immunological Mechanism", "url": "https://pubmed.ncbi.nlm.nih.gov/33673048/", "abstract": "A vaccine must elicit efficient adaptive immunity including B and T cell responses.", "text": "To protect against pathogen infection a vaccine must elicit efficient adaptive immunity including B and T cell responses. While B cell responses are key as they can mediate antibody-dependent protection T cells can modulate B cell activity and directly contribute to the elimination of pathogen-infected cells. mRNA vaccines represent a promising platform for infectious disease prevention due to their ability to rapidly encode any antigen of interest and stimulate both humoral and cellular immune responses."},
    {"id": "paper_03", "field": "medicine", "title": "Molecular Mechanisms of Antibiotic Resistance Revisited", "url": "https://pubmed.ncbi.nlm.nih.gov/36411397/", "abstract": "Antibiotic resistance is a global health emergency with resistance detected to all antibiotics in clinical use.", "text": "Antibiotic resistance is a global health emergency with resistance detected to all antibiotics currently in clinical use and only a few novel drugs in the pipeline. Understanding the molecular mechanisms that bacteria use to resist the action of antimicrobials is critical to recognize global patterns of resistance and to improve the use of current drugs as well as for the design of new drugs less susceptible to resistance development."},
    {"id": "paper_04", "field": "medicine", "title": "Global Burden of Bacterial Antimicrobial Resistance in 2019", "url": "https://pubmed.ncbi.nlm.nih.gov/35065702/", "abstract": "Antimicrobial resistance is a major global health threat causing millions of deaths annually.", "text": "Antimicrobial resistance is a major cause of death worldwide with the number of deaths attributable to bacterial antimicrobial resistance being substantial across all world regions. The study estimated the global burden of antimicrobial resistance using predictive statistical modelling to produce estimates for all locations. Resistance to antibiotics was found across a wide range of bacterial pathogens and infection types."},
    {"id": "paper_05", "field": "biology", "title": "CRISPR Technology A Decade of Genome Editing", "url": "https://pubmed.ncbi.nlm.nih.gov/36656942/", "abstract": "CRISPR-Cas9 has transformed biological research and medicine over the past decade.", "text": "CRISPR-Cas9 has become one of the most powerful tools in biology and medicine since its development as a genome editing platform. The technology allows scientists to make precise changes to DNA sequences in virtually any organism. Applications include correcting disease-causing mutations engineering disease-resistant crops developing new model organisms for research and creating potential therapies for genetic diseases."},
    {"id": "paper_06", "field": "biology", "title": "CRISPR Cas9 Gene Editing in Hematological Disorders", "url": "https://pubmed.ncbi.nlm.nih.gov/36610813/", "abstract": "CRISPR Cas9 gene editing shows promise for treating hematological disorders.", "text": "Gene therapy using CRISPR Cas9 has shown promise for treating hematological disorders including sickle cell disease and beta-thalassemia. The approach involves editing hematopoietic stem and progenitor cells to correct disease-causing mutations or to reactivate fetal hemoglobin expression. Clinical trials have demonstrated encouraging results with some patients achieving transfusion independence following treatment."},
    {"id": "paper_07", "field": "medicine", "title": "Memory CD8 T Cell Diversity Following mRNA Vaccination", "url": "https://pubmed.ncbi.nlm.nih.gov/36138186/", "abstract": "High responders to mRNA vaccination showed enhanced antibody neutralizing activity.", "text": "Understanding immune responses to SARS-CoV-2 messenger RNA vaccines is important for improving vaccine design and predicting protection. Analysis of B cell and T cell memory programs showed significant variability between individuals classified as high and low responders. High responders were characterized by enhanced antibody-neutralizing activity increased frequency of central memory T cells and durable spike-specific CD8 T cell responses."},
    {"id": "paper_08", "field": "medicine", "title": "Multidrug-Resistant Bacteria Mechanisms and Prophylaxis", "url": "https://pubmed.ncbi.nlm.nih.gov/36105930/", "abstract": "Multidrug resistance in bacteria has become a critical public health concern.", "text": "Resistance to antibiotics is one of the crucial issues related to public health. Earlier such resistance was limited to nosocomial infections but it has now become a common phenomenon across community settings. Several factors including overexploitation of antibiotics excessive application of broad-spectrum drugs and a shortage of target-oriented antimicrobial drugs contribute to this condition."},
    {"id": "paper_09", "field": "medicine", "title": "mRNA Vaccines Durable Immune Memory to SARS-CoV-2", "url": "https://pubmed.ncbi.nlm.nih.gov/34648302/", "abstract": "mRNA vaccines induce robust and durable cellular immune memory to SARS-CoV-2.", "text": "Recall responses to vaccination in individuals with preexisting immunity primarily increased antibody levels without substantially altering antibody decay rates. These findings demonstrate robust cellular immune memory to SARS-CoV-2 following mRNA vaccination. Both spike-specific CD4 and CD8 T cell responses were detectable months after vaccination and memory B cells continued to mature over time."},
    {"id": "paper_10", "field": "medicine", "title": "Antibiotic Resistance Challenges and Emerging Strategies", "url": "https://pubmed.ncbi.nlm.nih.gov/35949048/", "abstract": "Antibiotic resistance poses a global health threat requiring new antimicrobial strategies.", "text": "Antibiotic resistance has emerged as a major global threat to public health with resistant infections becoming increasingly difficult to treat across all clinical settings. The rise of multidrug-resistant organisms threatens to undermine decades of medical advances. Addressing this challenge requires a multifaceted approach including the development of new antimicrobial agents improved diagnostic tools enhanced infection prevention measures and international coordination on surveillance programs."},
    {"id": "paper_11", "field": "medicine", "title": "Asthma Medications and Management", "url": "https://pubmed.ncbi.nlm.nih.gov/30285350/", "abstract": "Asthma is a chronic inflammatory illness impacting millions daily treated with beta-2 agonists inhaled corticosteroids and other medications.", "text": "Asthma is a wide-reaching chronic inflammatory illness that impacts millions of people daily. It is frequently responsible for unscheduled healthcare usage missed school and workdays. It is an inappropriate immune response to a triggering factor that induces bronchial hyperreactivity constriction with remodeling of smooth muscle and increased mucous secretion into the airways. Several classifications of medications are utilized to treat and manage chronic asthma including beta-2 agonists anticholinergics and inhaled corticosteroids."},
    {"id": "paper_12", "field": "medicine", "title": "Asthma Management in Adults", "url": "https://pubmed.ncbi.nlm.nih.gov/36283607/", "abstract": "Management of asthma in adults has advanced significantly with new biologics improving treatment for severe asthma.", "text": "Management of asthma in adults has advanced in the past 10 years. Central to these advances has been further clarification of type 2 mechanisms of airway inflammation and utilization of biomarkers including eosinophils and fractional exhaled nitric oxide. Five new biologics were approved to revolutionize severe asthma treatment. These biologics significantly prevent exacerbations and spare systemic corticosteroids use. Guidelines support the effectiveness of inhaled corticosteroids with long-acting beta agonists for both maintenance and rescue therapy."},
    {"id": "paper_13", "field": "medicine", "title": "Advances in the Diagnosis and Treatment of Sickle Cell Disease", "url": "https://pubmed.ncbi.nlm.nih.gov/35241123/", "abstract": "Sickle cell disease affects approximately 100000 individuals in the USA caused by mutations in the beta globin gene.", "text": "Sickle cell disease affects approximately 100000 individuals in the USA and more than 3 million worldwide and is caused by mutations in the beta globin gene that result in sickle hemoglobin production. Sickle hemoglobin polymerization leads to red blood cell sickling chronic hemolysis and vaso-occlusion. Acute and chronic pain as well as end-organ damage occur throughout the lifespan of individuals living with sickle cell disease resulting in significant morbidity and a median life expectancy of 43 years in the USA."},
    {"id": "paper_14", "field": "medicine", "title": "Development of Curative Therapies for Sickle Cell Disease", "url": "https://pubmed.ncbi.nlm.nih.gov/36507504/", "abstract": "Disease modifying therapies such as hydroxyurea and crizanlizumab reduce pain crises while gene therapy represents an emerging curative approach.", "text": "Recent advances in managing sickle cell disease have significantly improved patient survival and quality of life. Disease modifying drug therapies such as hydroxyurea L-glutamine voxelotor and crizanlizumab reduce pain crises and severe complications. Allogeneic hematopoietic stem cell transplantation using matched sibling donors is currently the only standard curative option however only a small proportion of patients have such donors. Gene therapy approaches using lentiviral vectors and CRISPR gene editing are emerging as promising curative strategies."},
    {"id": "paper_15", "field": "medicine", "title": "Updates in Hypertension New Trials and Treatment Targets", "url": "https://pubmed.ncbi.nlm.nih.gov/35249970/", "abstract": "Recent trials support intensive blood pressure lowering to 110-130 mmHg in older patients and identify new pharmacological strategies.", "text": "Several recent trials and observational studies have identified critical areas that can help to improve the management and measurement of blood pressure in patients with hypertension. High quality trial evidence supports intensive systolic blood pressure lowering to 110-130 mmHg in older patients and potassium based salt substitution in patients without chronic kidney disease. New pharmacological approaches to treat resistant hypertension are being developed and device based approaches including renal denervation are also being evaluated."},
    {"id": "paper_16", "field": "medicine", "title": "WHO Guideline for Pharmacological Treatment of Hypertension", "url": "https://pubmed.ncbi.nlm.nih.gov/34775787/", "abstract": "The World Health Organization guideline provides evidence based recommendations for pharmacological treatment of hypertension in adults.", "text": "Hypertension is one of the leading causes of cardiovascular disease morbidity and mortality worldwide. The World Health Organization guideline on pharmacological treatment of hypertension in adults provides recommendations based on systematic reviews of evidence. First line antihypertensive agents include thiazide diuretics calcium channel blockers ACE inhibitors and angiotensin receptor blockers. The guideline emphasizes the importance of lifestyle modifications alongside pharmacological treatment."},
    {"id": "paper_17", "field": "medicine", "title": "Hypothyroidism Diagnosis and Evidence-Based Treatment", "url": "https://pubmed.ncbi.nlm.nih.gov/35384263/", "abstract": "Hypothyroidism affects up to 5 percent of the global population and is managed primarily with levothyroxine replacement therapy.", "text": "Hypothyroidism affects up to 5 percent of the global population. Incidence increases with age and is more common in women. Symptoms can develop slowly and often mimic symptoms of other disorders including menstrual cycle abnormalities. Diagnosis relies on testing of thyroid stimulating hormone levels and confirmation with thyroxine levels. Management usually involves monotherapy with levothyroxine taken on an empty stomach."},
    {"id": "paper_18", "field": "medicine", "title": "Evaluating Health Outcomes in the Treatment of Hypothyroidism", "url": "https://pubmed.ncbi.nlm.nih.gov/36329885/", "abstract": "Clinical hypothyroidism requires lifelong thyroid hormone replacement with the primary goal of restoring normal thyroid function.", "text": "Clinical hypothyroidism is defined by the inadequate production of thyroid hormone from the thyroid gland to maintain normal organ system functions. For nearly all patients with clinical hypothyroidism lifelong treatment with thyroid hormone replacement is required. The primary goal of treatment is to provide the appropriate daily dose of thyroid hormone to restore normal thyroid function for each individual patient. Normalization of thyrotropin level is the primary measure of effectiveness of treatment."},
    {"id": "paper_19", "field": "medicine", "title": "Management and Metabolic Characterization of Hyperthyroidism and Hypothyroidism", "url": "https://pubmed.ncbi.nlm.nih.gov/36455479/", "abstract": "Hyperthyroidism and hypothyroidism are common thyroid disorders with traditional treatments including hormone replacement and antithyroid drugs.", "text": "Hyperthyroidism and hypothyroidism are common diseases resulting from thyroid dysfunction and are simple to diagnose and treat. The traditional treatment for hypothyroidism is thyroid hormone replacement therapy. The traditional treatments for hyperthyroidism include antithyroid drugs iodine radiotherapy and surgery. Insufficient treatment can result in long-term thyroid hormone deficiency associated with increased risk of cardiovascular disease while overtreatment can result in heart disease and osteoporosis."},
    {"id": "paper_20", "field": "medicine", "title": "Updated Sexually Transmitted Infections Guidelines", "url": "https://pubmed.ncbi.nlm.nih.gov/37427972/", "abstract": "Growing antimicrobial resistance in gonorrhea and chlamydia has driven the need to update STI treatment guidelines.", "text": "One of the most persistent public health concerns continues to be sexually transmitted infections and their consequences. A large portion of sexually transmitted infections occur in adolescents and young adults with serious consequences such as infertility and systemic disease. There has been growing evidence for antimicrobial resistance in strains of gonorrhea and chlamydia which has provided the need to update treatment guidelines to prevent continued resistance and decrease the rate of treatment failure."},
    {"id": "paper_21", "field": "medicine", "title": "Diagnosis and Treatment of Sexually Transmitted Infections A Review", "url": "https://pubmed.ncbi.nlm.nih.gov/35015033/", "abstract": "Sexually transmitted infections including chlamydia gonorrhea syphilis herpes and HPV require accurate diagnosis and appropriate treatment.", "text": "Sexually transmitted infections including chlamydia gonorrhea syphilis herpes simplex virus and human papillomavirus are among the most common infectious diseases worldwide. Accurate diagnosis using nucleic acid amplification tests is the standard of care for chlamydia and gonorrhea. Treatment regimens vary by pathogen and must account for increasing antimicrobial resistance patterns particularly for gonorrhea. Partner notification and treatment are essential components of management to prevent reinfection and reduce community transmission."},
    {"id": "paper_22", "field": "medicine", "title": "Asthma Clinical Trials Update 2023", "url": "https://pubmed.ncbi.nlm.nih.gov/36243545/", "abstract": "Asthma is a significant worldwide health issue with biologics now available to treat severe type 2 high asthma.", "text": "Asthma is a complex heterogeneous chronic airway disease with high prevalence of uncontrolled disease. New therapies including biologics are now available to treat type 2 high asthma characterized by eosinophilic inflammation. Treatment of type 2 low asthma remains a challenge with limited effective therapeutic options. Biologics have shown promising results and the potential for changing the treatment of uncontrolled asthma."},
    {"id": "paper_23", "field": "medicine", "title": "Sickle Cell Disease in the New Era Advances in Drug Treatment", "url": "https://pubmed.ncbi.nlm.nih.gov/36096995/", "abstract": "Four currently approved drugs for sickle cell disease including hydroxyurea L-glutamine voxelotor and crizanlizumab are improving outcomes.", "text": "This review provides an overview of therapeutic strategies for sickle cell disease and discusses the four currently approved drugs in detail including hydroxyurea L-glutamine voxelotor and crizanlizumab. Each of these agents targets different aspects of sickle cell disease pathophysiology including fetal hemoglobin induction oxidative stress reduction red blood cell sickling and vaso-occlusion. Ongoing clinical trials are evaluating new drugs and drug combinations and gene therapy represents the next frontier in potentially curative treatment."},
    {"id": "paper_24", "field": "medicine", "title": "Arterial Hypertension Clinical Trials Update 2023", "url": "https://pubmed.ncbi.nlm.nih.gov/37443261/", "abstract": "The 2022 and 2023 hypertension clinical trials summarize new pharmacological approaches for resistant hypertension and device based treatment strategies.", "text": "Arterial hypertension is associated with increased morbidity and mortality and research in the field is highly dynamic. This summary reviews the most important clinical trials published in 2022 and early 2023. Findings on new pharmacological approaches to treat resistant hypertension are presented and new knowledge about the optimal timing of antihypertensive medication intake is discussed. Novel clinical data on device based approaches to treat hypertension including renal denervation are also summarized."},
    {"id": "paper_25", "field": "medicine", "title": "Burden of Chlamydia Gonorrhea and Syphilis in Older Adults", "url": "https://pubmed.ncbi.nlm.nih.gov/36626249/", "abstract": "Sexually transmitted infections among older adults are understudied with prevalence ranges highlighting a growing public health concern.", "text": "Increases in life expectancy and changes in sexual partnering suggest that sexually transmitted infections among older persons could be on the rise yet there have been relatively few studies examining sexually transmitted infections in this demographic. A systematic review aimed to further characterize the incidence and prevalence of chlamydia gonorrhea and syphilis along with associated risk factors among older adults aged 45 years or older in the United States. The review found prevalence ranges of syphilis from 0 to 18 percent chlamydia from 0 to 14 percent and gonorrhea from 0 to 15 percent."}
]

def load_papers():
    prepared = []
    for paper in papers:
        cleaned = clean_text(paper["text"])
        truncated = truncate_text(cleaned)
        prepared.append({
            "id": paper["id"],
            "field": paper["field"],
            "title": paper["title"],
            "url": paper["url"],
            "abstract": paper["abstract"],
            "input_text": truncated
        })
    print(f"Loaded {len(prepared)} papers successfully!")
    return prepared

papers_loaded = load_papers()
print("Papers ready!")

Loaded 25 papers successfully!
Papers ready!


In [4]:
import os

def save_output(content, filename, output_dir="outputs/"):
    os.makedirs(output_dir, exist_ok=True)
    filepath = os.path.join(output_dir, filename)
    with open(filepath, 'w') as f:
        f.write(content)
    print(f"Output saved to {filepath}")
    return filepath

def summarize_bart(text, tokenizer, model):
    inputs = tokenizer(text, return_tensors="pt", max_length=1024, truncation=True)
    output = model.generate(**inputs, max_length=150, min_length=30)
    return tokenizer.decode(output[0], skip_special_tokens=True)

def summarize_t5(text, tokenizer, model):
    input_text = "summarize: " + text
    inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True)
    output = model.generate(**inputs, max_length=150, min_length=30)
    return tokenizer.decode(output[0], skip_special_tokens=True)

results = []
output_text = "BART vs T5 Summarization Results - 25 Papers\n"
output_text += "=" * 60 + "\n\n"

for paper in papers_loaded:
    print(f"Processing: {paper['title']}...")
    bart_summary = summarize_bart(paper["input_text"], bart_tokenizer, bart_model)
    t5_summary = summarize_t5(paper["input_text"], t5_tokenizer, t5_model)

    results.append({
        "id": paper["id"],
        "title": paper["title"],
        "field": paper["field"],
        "url": paper["url"],
        "abstract": paper["abstract"],
        "bart_summary": bart_summary,
        "t5_summary": t5_summary
    })

    output_text += f"Paper ID: {paper['id']}\n"
    output_text += f"Title: {paper['title']}\n"
    output_text += f"Field: {paper['field']}\n"
    output_text += f"Source: {paper['url']}\n"
    output_text += f"Reference Abstract: {paper['abstract']}\n"
    output_text += f"BART Summary: {bart_summary}\n"
    output_text += f"T5 Summary: {t5_summary}\n"
    output_text += "-" * 60 + "\n\n"

    print(f"BART: {bart_summary[:80]}...")
    print(f"T5:   {t5_summary[:80]}...")

save_output(output_text, "samples.txt")
print("\nAll 25 papers processed!")

Processing: Development of mRNA Vaccines and Their Delivery System...
BART: The rapid development of mRNA vaccines contributed to managing the COVID-19 pand...
T5:   Molecular RNA vaccines are a promising vaccine for the COVID-19 pandemic. They a...
Processing: SARS-CoV-2 mRNA Vaccines Immunological Mechanism...
BART:  mRNA vaccines represent a promising platform for infectious disease prevention....
T5:   mRNA vaccines are a promising platform for infectious disease prevention. They c...
Processing: Molecular Mechanisms of Antibiotic Resistance Revisited...
BART: Antibiotic resistance is a global health emergency with resistance detected to a...
T5:   Understanding the molecular mechanisms that bacteria use to resist the action of...
Processing: Global Burden of Bacterial Antimicrobial Resistance in 2019...
BART: Antimicrobial resistance is a major cause of death worldwide. The study estimate...
T5:   A global burden of antimicrobial resistance is estimated using predictive statis...


In [5]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

rouge_text = "ROUGE Evaluation Results - BART vs T5\n"
rouge_text += "=" * 60 + "\n\n"

bart_r1_scores = []
bart_r2_scores = []
bart_rL_scores = []
t5_r1_scores = []
t5_r2_scores = []
t5_rL_scores = []

for result in results:
    bart_scores = scorer.score(result["abstract"], result["bart_summary"])
    t5_scores = scorer.score(result["abstract"], result["t5_summary"])

    bart_r1_scores.append(bart_scores['rouge1'].fmeasure)
    bart_r2_scores.append(bart_scores['rouge2'].fmeasure)
    bart_rL_scores.append(bart_scores['rougeL'].fmeasure)
    t5_r1_scores.append(t5_scores['rouge1'].fmeasure)
    t5_r2_scores.append(t5_scores['rouge2'].fmeasure)
    t5_rL_scores.append(t5_scores['rougeL'].fmeasure)

    rouge_text += f"Paper: {result['title']}\n"
    rouge_text += f"BART - ROUGE-1: {bart_scores['rouge1'].fmeasure:.3f} | ROUGE-2: {bart_scores['rouge2'].fmeasure:.3f} | ROUGE-L: {bart_scores['rougeL'].fmeasure:.3f}\n"
    rouge_text += f"T5   - ROUGE-1: {t5_scores['rouge1'].fmeasure:.3f} | ROUGE-2: {t5_scores['rouge2'].fmeasure:.3f} | ROUGE-L: {t5_scores['rougeL'].fmeasure:.3f}\n"
    rouge_text += "-" * 60 + "\n"

rouge_text += "\n=== AVERAGE SCORES ACROSS ALL 25 PAPERS ===\n"
rouge_text += f"BART - ROUGE-1: {sum(bart_r1_scores)/len(bart_r1_scores):.3f} | ROUGE-2: {sum(bart_r2_scores)/len(bart_r2_scores):.3f} | ROUGE-L: {sum(bart_rL_scores)/len(bart_rL_scores):.3f}\n"
rouge_text += f"T5   - ROUGE-1: {sum(t5_r1_scores)/len(t5_r1_scores):.3f} | ROUGE-2: {sum(t5_r2_scores)/len(t5_r2_scores):.3f} | ROUGE-L: {sum(t5_rL_scores)/len(t5_rL_scores):.3f}\n"

print(rouge_text)
save_output(rouge_text, "rouge_results.txt")
print("ROUGE scores saved!")

ROUGE Evaluation Results - BART vs T5

Paper: Development of mRNA Vaccines and Their Delivery System
BART - ROUGE-1: 0.371 | ROUGE-2: 0.353 | ROUGE-L: 0.371
T5   - ROUGE-1: 0.486 | ROUGE-2: 0.343 | ROUGE-L: 0.270
------------------------------------------------------------
Paper: SARS-CoV-2 mRNA Vaccines Immunological Mechanism
BART - ROUGE-1: 0.256 | ROUGE-2: 0.000 | ROUGE-L: 0.154
T5   - ROUGE-1: 0.256 | ROUGE-2: 0.000 | ROUGE-L: 0.154
------------------------------------------------------------
Paper: Molecular Mechanisms of Antibiotic Resistance Revisited
BART - ROUGE-1: 0.593 | ROUGE-2: 0.538 | ROUGE-L: 0.593
T5   - ROUGE-1: 0.261 | ROUGE-2: 0.000 | ROUGE-L: 0.217
------------------------------------------------------------
Paper: Global Burden of Bacterial Antimicrobial Resistance in 2019
BART - ROUGE-1: 0.360 | ROUGE-2: 0.208 | ROUGE-L: 0.320
T5   - ROUGE-1: 0.389 | ROUGE-2: 0.118 | ROUGE-L: 0.278
------------------------------------------------------------
Paper: CRISPR Technol

In [6]:
from google.colab import files
files.download("outputs/samples.txt")
files.download("outputs/rouge_results.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
with open("outputs/samples.txt", "r") as f:
    print(f.read())

BART vs T5 Summarization Results - 25 Papers

Paper ID: paper_01
Title: Development of mRNA Vaccines and Their Delivery System
Field: medicine
Source: https://pubmed.ncbi.nlm.nih.gov/36697236/
Reference Abstract: The rapid development of mRNA vaccines contributed to managing the COVID-19 pandemic.
BART Summary: The rapid development of mRNA vaccines contributed to managing the COVID-19 pandemic and may help manage future outbreaks of infectious diseases. Because the antigens targeted by mRNA vaccines can be easily altered by simply changing the sequence present in the coding region of mRNA structures it is more appropriate to develop vaccines during rapidly developing outbreaks.
T5 Summary: Molecular RNA vaccines are a promising vaccine for the COVID-19 pandemic. They are a promising vaccine for the rapid development of infectious diseases.
------------------------------------------------------------

Paper ID: paper_02
Title: SARS-CoV-2 mRNA Vaccines Immunological Mechanism
Field: med